In [ ]:
import vaex
import pandas as pd
from astroquery.gaia import Gaia
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcdefaults()  
from matplotlib.lines import Line2D
plt.rcParams['text.usetex'] = True
from config import DATA_DIR

In [ ]:
#li + alpha abundances
cat = vaex.open(DATA_DIR/'li_alpha.hdf5')
cat = cat.to_pandas_df()
cat['source_id'] = cat['source_id'].astype(str)

In [ ]:
cs = pd.read_csv('../data/true_sources_basta_output_final.csv',dtype={'source_id':str})
halo = pd.read_csv('../data/kinematic_halo.csv',dtype={'source_id':str})

cs['age'] = cs['age']/1000 #for plotting purposes

In [ ]:
#merge to get alpha abundances
cs = cs.merge(cat[['source_id','aom_xp','e_aom_xp','moh_xp']], on='source_id', how='left') 
len(cs[~cs['aom_xp'].isna()])

In [ ]:
cs_halo = cs[cs['source_id'].isin(halo['source_id'])]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Normalise G column for colour mapping
norm = plt.Normalize(vmin=cs['age'].min(), vmax=cs['age'].max())
cmap = 'plasma_r'
sc = ax.scatter(cs['FeH_input'], cs['aom_xp'],
                c=cs['age'], cmap=cmap, norm=norm,
             s=100, zorder=1,edgecolor='black', linewidth=0.5,rasterized=True)
ax.scatter(cs_halo['FeH_input'],cs_halo['aom_xp'],c=cs_halo['age'], cmap=cmap, norm=norm,
             s=1000, marker='*',edgecolor='lime', linewidth=2,rasterized=True)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('$\mathrm{Age\,\,[Gyr]}$', fontsize=25)
cbar.ax.tick_params(labelsize=16)

ax.set_xlabel('$\mathrm{[Fe / H]}$', fontsize=25)
ax.set_ylabel('$\mathrm{[\\alpha / Fe]}$', fontsize=25)
ax.tick_params(axis='both', which='major', labelsize=20, pad=5, size=8)

feh = np.linspace(-3.3, 0.6, 1000)
threshold = np.where(
    feh < -0.4, 0.14,
    np.where(
        feh > 0.2, 0.05,
        0.08 - 0.15 * feh
    )
)
legend_handles = [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='white',
           markeredgecolor='limegreen', markeredgewidth=2, markersize=15, label=r'$\mathrm{Kinematic\ Halo}$'),
]

ax.legend(handles=legend_handles, fontsize=20, markerscale=1.5,facecolor='white', framealpha=1,loc='lower left')

ax.plot(feh, threshold, color='blue', linestyle='--', linewidth=3, zorder=2,
        label=r'$\mathrm{\alpha - rich \,/ \, \alpha - poor \,\,division}$')

plt.savefig('../figures/alpha_fe_age_divide_kin_halo.pdf', format='pdf', bbox_inches='tight')
plt.show()